# ShopDesk, Section 2 Lab 2: Sessions - Resume and Fork

A beginner-friendly notebook on **session persistence**. First the **Claude Code CLI**
workflow (`--resume`, `--fork-session`) that you run in a terminal inside a Git repo, then a
runnable **Claude Agent SDK** demonstration of the same ideas so you can watch context carry
across turns and branch on a fork. Runs **Sonnet** (`claude-sonnet-4-6`) through your
**Anthropic API key**.

## The real-world scenario

A ShopDesk investigation rarely finishes in one sitting. You close the terminal, come back
tomorrow, and you do not want to re-explain which order you were chasing. **Sessions** solve
this: the conversation is saved to disk, so you can **resume** exactly where you left off, or
**fork** it to try a different fix without losing the thread you were on.

The question this lab answers: **how do you continue prior work with full context, and how
do you branch it for a parallel attempt?**

## Objectives

- Use the Claude Code CLI to **name**, **continue**, and **resume** a session, and to
  **fork** one with `--fork-session`, inside a Git repository.
- Do the same programmatically with the Agent SDK using `resume` and `fork_session`.
- **Compare** a resumed session (keeps prior context) against a fresh one (starts empty),
  and use a fork for parallel exploration.

## What you'll observe

- A resumed session answers a back-reference ("that same order") correctly, because it
  still has the earlier turns.
- A fresh session cannot: with no history, it does not know which order you mean.
- A forked session starts from a copy of the original's history and diverges, while the
  original stays intact.

## How to run

The CLI section is **terminal** commands: run them in a real shell inside a Git repo with
Claude Code installed; they are shown here for reference, not executed by the notebook. The
Agent SDK section is **live**: paste a real key into **Setup 2/3** and re-run from the top,
otherwise it skips. **Node.js 18+** must be installed for the Agent SDK.

## 0. Setup

**This cell:** installs the packages for the runnable Agent SDK section. The CLI
section needs **Claude Code** installed separately in your terminal; the Agent SDK here needs
Node.js 18+, which cannot be pip-installed.

In [ ]:
# ===== SETUP 1/3 - install the Agent SDK =====
%pip install -q claude-agent-sdk anthropic python-dotenv

**This cell:** imports what we need, pins the model, and sets a `RUN_LIVE` switch so
the Agent SDK calls fire only with a real key.

In [ ]:
# ===== SETUP 2/3 - imports, the model, and a live/offline switch =====
import os                                       # read the API key from the environment
import sys                                      # detect Windows (it needs a special event loop)
import asyncio                                  # the Agent SDK is async; we drive it ourselves
import threading                                # run that async loop in a side thread (notebook-safe)

try:                                            # load a .env file if present
    from dotenv import load_dotenv              #   import the loader
    load_dotenv()                               #   read .env into environment variables
except Exception:                               # not installed? that is fine
    pass                                        #   set the key another way

MODEL = "claude-sonnet-4-6"                      # the Sonnet model every call will use

os.environ.setdefault("ANTHROPIC_API_KEY", "sk-ant-...")     # placeholder unless you set a real key
_key = os.environ["ANTHROPIC_API_KEY"]           # read whatever key is set
RUN_LIVE = _key.startswith("sk-ant-") and _key != "sk-ant-..."   # True only for a real key
print("live model calls:", "ON" if RUN_LIVE else "OFF (using a placeholder key)")

**This cell:** defines `run_async()`, a notebook-safe wrapper that runs any async Agent
SDK call in its own thread and event loop. We build it once so the session cells below can be
called like ordinary functions.

In [ ]:
# ===== SETUP 3/3 - a notebook-safe runner for async Agent SDK calls =====
def run_async(make_coro):                         # make_coro: a function returning a coroutine
    box = {}                                      #   carries the result or error out of the thread
    def worker():                                 #   runs in its own thread
        if sys.platform == "win32":               #     Windows needs the Proactor loop...
            loop = asyncio.ProactorEventLoop()    #       ...to spawn the Agent SDK subprocess
        else:                                     #     macOS / Linux:
            loop = asyncio.new_event_loop()       #       a plain new loop is fine
        asyncio.set_event_loop(loop)              #     make it this thread's loop
        try:                                      #
            box["value"] = loop.run_until_complete(make_coro())   # run the coroutine to completion
        except Exception as e:                    #     capture any error...
            box["error"] = e                      #       ...to re-raise on the main thread
        finally:                                  #
            loop.close()                          #     always close the loop
    t = threading.Thread(target=worker); t.start(); t.join()   # run it and wait
    if "error" in box:                            #   worker failed?
        raise box["error"]                        #     surface the error here
    return box.get("value")                       #   hand back the result

### How sessions work

A **session** is the conversation history the tool saves to disk as you work: your prompts,
every tool call and result, and every response. Because it persists, you can return to it
later with full context.

- **Continue** picks up the most recent session in the current directory. You track nothing.
- **Resume** picks up a specific session by id or name. You track the id; needed when you
  have several.
- **Fork** makes a *new* session that starts with a copy of the original's history, then
  diverges. The original stays unchanged, so you can explore an alternative safely.

Sessions are stored per project directory (under `~/.claude/projects/`) and are scoped to
that directory and its Git worktrees, so resume from the same folder you started in.

---

### 🎯 Lab objective - continue work, and branch it

**What you build:** the CLI commands for naming, resuming, and forking sessions, then a
runnable Agent SDK demo that resumes a ShopDesk thread and forks it.

**Why it helps you build real solutions:** long tasks span many sittings and often need a
"what if we tried it another way" branch. Sessions give you both without re-explaining
context or losing your place.

**How you'll see it:** a resumed session resolves a back-reference the fresh session cannot,
and a fork branches off while the original stays intact.

**This section is terminal commands**, meant for a real shell in your Git repo (not run
by the notebook). Start by **naming** a session so it is easy to find, then **continue** or
**resume** it later. Run these from the project directory the session belongs to.

```bash
# start a NEW named session in your repo (run from the project directory)
claude --name shopdesk-refund-bug

# later: continue the MOST RECENT session in this directory (fastest)
claude --continue

# or resume a SPECIFIC session by name or id (when you have several)
claude --resume shopdesk-refund-bug
claude --resume 59d46c3a-4fc1-4213-b8fa-d99925c0443b
```

Inside a running session you can also type `/rename <name>` to name it, or `/resume` to open
a picker and jump to another thread without leaving Claude Code.

**Forking (terminal commands).** To try an alternate approach without disturbing your
current thread, fork: combine `--resume` (or `--continue`) with `--fork-session`. The fork
gets a new session id; the original is untouched and still in the picker.

```bash
# branch the named session into a NEW session id, leaving the original intact
claude --resume shopdesk-refund-bug --fork-session

# from inside a running session, the same thing:
#   /branch alt-approach     fork here into a named branch
#   /resume                  return to the original from the picker
```

Because sessions are scoped to the directory, isolate truly parallel work with Git worktrees:
`git worktree add ../shopdesk-alt` then run a separate session in that folder. For automation,
capture the id with `claude -r <name> -p "..." --output-format json` and read `.session_id`,
rather than relying on `--continue`.

**This cell:** imports the Agent SDK session pieces and defines `ask()`, which runs one
turn with the given options and returns the session id plus the answer text. We read the id
from the `ResultMessage`, because `resume` and `fork_session` both need it.

In [ ]:
# ===== a one-turn helper that also captures the session id =====
from claude_agent_sdk import (                     # the Agent SDK pieces we use:
    query, ClaudeAgentOptions,                     #   run + options (resume / fork live here)
    AssistantMessage, ResultMessage, TextBlock,    #   message + block types
)

async def ask(prompt, options):                    # run one turn -> (session_id, answer_text)
    sid, answer = None, ""                          #   defaults
    async for message in query(prompt=prompt, options=options):   # stream the turn
        if isinstance(message, AssistantMessage):  #     the model spoke
            for block in message.content:          #       walk its blocks
                if isinstance(block, TextBlock):   #       keep the latest text
                    answer = block.text
        elif isinstance(message, ResultMessage):    #     the turn finished
            sid = getattr(message, "session_id", None)   #   grab the id for resume / fork
    return sid, answer                              #   hand both back

**This cell:** **turn 1** starts a fresh session that establishes context: it tells the
agent we are working on order A1. We capture the returned `session_id`, which is the handle we
will resume and fork from.

In [ ]:
# ===== turn 1: establish context and capture the session id =====
BASE = ClaudeAgentOptions(                          # a plain session, no resume yet
    model=MODEL,
    system_prompt="You are ShopDesk support. Track the order the user is investigating.")

if RUN_LIVE:                                        # needs a real key (and Node.js 18+)
    sid, ans = run_async(lambda: ask(                #   run turn 1
        "I am investigating order A1, which has shipped. Note that for me.", BASE))
    print("session id:", sid)                        #   the handle for resume / fork
    print("turn 1 answer:", ans)
else:
    sid = None                                       #   nothing to resume offline
    print("[skipped - set ANTHROPIC_API_KEY to run this live]")

**This cell:** **resume** that session and ask a back-reference question ("that same
order"). Because resume reloads the prior turns, the agent still knows we mean A1 and answers
without being told again.

In [ ]:
# ===== resume: the context carries over =====
if RUN_LIVE and sid:                                # need a real key and a captured id
    RESUMED = ClaudeAgentOptions(model=MODEL, resume=sid)   # continue the SAME session by id
    _, ans = run_async(lambda: ask("What was the status of that same order?", RESUMED))
    print("resumed answer:", ans)                    #   expected: it knows the order is A1, shipped
else:
    print("[skipped] expected: the resumed session recalls A1 and reports it shipped.")

**This cell:** the **comparison**: ask the *same* back-reference question in a brand-new
session with no history. With nothing to resume, the agent cannot know which order "that"
refers to. This is the difference resume makes.

In [ ]:
# ===== fresh: no history, so the back-reference fails =====
if RUN_LIVE:                                        # needs a real key
    _, ans = run_async(lambda: ask("What was the status of that same order?", BASE))   # new session
    print("fresh answer:", ans)                      #   expected: it asks which order you mean
else:
    print("[skipped] expected: the fresh session has no context and must ask which order.")

**This cell:** **fork** the original session to explore an alternative, using
`fork_session=True`. The fork copies the history up to now and then diverges; the original
`sid` is untouched, so you can always return to it. This is parallel exploration in code.

In [ ]:
# ===== fork: branch for a parallel attempt, original stays intact =====
if RUN_LIVE and sid:                                # need a real key and a captured id
    FORKED = ClaudeAgentOptions(model=MODEL, resume=sid, fork_session=True)   # copy-then-diverge
    fork_sid, ans = run_async(lambda: ask(           #   explore a different direction on the branch
        "On a separate branch: draft a goodwill message about that order.", FORKED))
    print("fork session id (new):", fork_sid)        #   a NEW id; the original sid is unchanged
    print("original session id  :", sid)             #   still valid, still resumable
    print("fork answer:", ans)
else:
    print("[skipped] expected: a NEW session id, with the original left intact for later.")

| anti-pattern | what to do instead |
|---|---|
| leave sessions unnamed and hunt later | name them at the start (`--name`, or `/rename`) |
| resume from a different directory | resume from the same project folder the session started in |
| resume when you meant to branch | add `--fork-session` (or use `fork_session=True`) to keep the original |
| rely on `--continue` in a script | capture and resume an explicit session id for reliable automation |

**Lesson:** a session is your conversation, saved to disk. **Resume** brings back full
context so you never re-explain; a **fresh** session starts empty. **Fork** copies the thread
so you can try an alternative without risking the original. In the CLI these are `--resume`
and `--fork-session`; in the Agent SDK they are `resume` and `fork_session`.

---

## Recap - resume and fork

| Action | CLI | Agent SDK | Result |
|---|---|---|---|
| Continue latest | `claude --continue` | `continue_conversation=True` | picks up the most recent thread here |
| Resume specific | `claude --resume <id/name>` | `resume=session_id` | reloads that thread with full context |
| Fork | `claude --resume <id> --fork-session` | `resume=session_id, fork_session=True` | new id, copied history, original intact |
| Fresh | `claude` | new `ClaudeAgentOptions(...)` | empty context, nothing to recall |

One principle to carry forward: **resume to keep context, fork to explore safely, and always
work from the session's own directory.** To run the Agent SDK demo live, paste a real key
into **Setup 2/3** and re-run from the top. Then try it in your terminal: name a session,
make a change in your repo, resume it tomorrow, and fork it before a risky edit.